# Adidas US Sales — End-to-End ETL Pipeline (Python & MySQL)

## Project Overview

This project implements a complete Extract, Transform, Load (ETL) pipeline on the Adidas US Sales dataset. Raw data is extracted from an kaggle source file, cleaned and validated in Python using pandas, and loaded into a MySQL database using SQLAlchemy for downstream business analysis.

**Pipeline Flow:** Raw xlsx → Data Exploration → Data Cleaning → MySQL → Power BI

**Tools:** Python, pandas, NumPy, SQLAlchemy, pymysql, MySQL 

**Dataset:** 9,648 Adidas US Sales Transactions | Year: 2020–2021 | 13 columns

### 1. Extract & Importing Libraries
- Dataset: `Adidas US Sales Datasets.xlsx`
- Header located at row 5 (`header=4`); columns B through N loaded
- Loading the required Python libraries for data manipulation

In [1]:
# import zipfile
# new = zipfile.ZipFile('Adidas US Sales.zip')
# new.extractall()
# new.close()

In [2]:
import pandas as pd
import numpy as np
df = pd.read_excel('Adidas US Sales Datasets.xlsx', header=4, usecols="B:N")
df

,Retailer,Retailer ID,Invoice Date,Region,State,City,Product,Price per Unit,Units Sold,Total Sales,Operating Profit,Operating Margin,Sales Method
0,Foot Locker,1185732,2020-01-01,Northeast,New York,New York,Men's Street Footwear,50.0,1200,600000.0,300000.00,0.50,In-store
1,Foot Locker,1185732,2020-01-02,Northeast,New York,New York,Men's Athletic Footwear,50.0,1000,500000.0,150000.00,0.30,In-store
2,Foot Locker,1185732,2020-01-03,Northeast,New York,New York,Women's Street Footwear,40.0,1000,400000.0,140000.00,0.35,In-store
3,Foot Locker,1185732,2020-01-04,Northeast,New York,New York,Women's Athletic Footwear,45.0,850,382500.0,133875.00,0.35,In-store
4,Foot Locker,1185732,2020-01-05,Northeast,New York,New York,Men's Apparel,60.0,900,540000.0,162000.00,0.30,In-store
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9643,Foot Locker,1185732,2021-01-24,Northeast,New Hampshire,Manchester,Men's Apparel,50.0,64,3200.0,896.00,0.28,Outlet
9644,Foot Locker,1185732,2021-01-24,Northeast,New Hampshire,Manchester,Women's Apparel,41.0,105,4305.0,1377.60,0.32,Outlet
9645,Foot Locker,1185732,2021-02-22,Northeast,New Hampshire,Manchester,Men's Street Footwear,41.0,184,7544.0,2791.28,0.37,Outlet
9646,Foot Locker,1185732,2021-02-22,Northeast,New Hampshire,Manchester,Men's Athletic Footwear,42.0,70,2940.0,1234.80,0.42,Outlet


### 2. Exploratory Data Analysis (EDA) 
Understanding the structure, data types, missing value counts before any transformation.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9648 entries, 0 to 9647
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Retailer          9648 non-null   object        
 1   Retailer ID       9648 non-null   int64         
 2   Invoice Date      9648 non-null   datetime64[ns]
 3   Region            9648 non-null   object        
 4   State             9648 non-null   object        
 5   City              9648 non-null   object        
 6   Product           9648 non-null   object        
 7   Price per Unit    9648 non-null   float64       
 8   Units Sold        9648 non-null   int64         
 9   Total Sales       9648 non-null   float64       
 10  Operating Profit  9648 non-null   float64       
 11  Operating Margin  9648 non-null   float64       
 12  Sales Method      9648 non-null   object        
dtypes: datetime64[ns](1), float64(4), int64(2), object(6)
memory usage: 980.0+ KB


In [4]:
df.describe()

,Retailer ID,Invoice Date,Price per Unit,Units Sold,Total Sales,Operating Profit,Operating Margin
count,9.648000e+03,9648,9648.000000,9648.000000,9648.000000,9648.000000,9648.000000
mean,1.173850e+06,2021-05-10 15:20:44.776119296,45.216625,256.930037,93273.437500,34425.244761,0.422991
min,1.128299e+06,2020-01-01 00:00:00,7.000000,0.000000,0.000000,0.000000,0.100000
25%,1.185732e+06,2021-02-17 00:00:00,35.000000,106.000000,4254.500000,1921.752500,0.350000
50%,1.185732e+06,2021-06-04 00:00:00,45.000000,176.000000,9576.000000,4371.420000,0.410000
75%,1.185732e+06,2021-09-16 00:00:00,55.000000,350.000000,150000.000000,52062.500000,0.490000
max,1.197831e+06,2021-12-31 00:00:00,110.000000,1275.000000,825000.000000,390000.000000,0.800000
std,2.636038e+04,NaN,14.705397,214.252030,141916.016727,54193.113713,0.097197


In [5]:
df.duplicated().sum()

0

In [6]:
df.isnull().sum()

Retailer            0
Retailer ID         0
Invoice Date        0
Region              0
State               0
City                0
Product             0
Price per Unit      0
Units Sold          0
Total Sales         0
Operating Profit    0
Operating Margin    0
Sales Method        0
dtype: int64

### 3. Data Cleaning & Transformation

#### 3.1 Schema Standardisation
- All column names converted to `snake_case` using a lambda rename
- String and categorical columns assigned correct dtypes
- `retailer_id` cast to `string` — it is an identifier, not a numeric quantity
- Float columns rounded to 2 decimal places; `operating_margin` excluded
  intentionally because it is used in arithmetic before conversion

#### 3.2 Data Quality Findings

| Issues | Rows Affected | Action Taken |
|-------|--------------|--------------|
|Duplicate rows | 0 | None required |
|Null values in any column | 0 | None required |
|`units_sold = 0` | 4 | Removed — division-by-zero risk |
|`total_sales` ≠ `price_per_unit × units_sold` | 3,886 | Corrected — 10× wholesale pricing anomaly |
|`operating_profit` inconsistent with corrected sales | 3,886 | Recalculated from corrected `total_sales` |
|`operating_margin` stored as decimal (0.30) | All rows | Converted to percentage (30.0) after all arithmetic |

In [7]:
df = df.rename(columns=lambda col: col.strip().lower().replace(" ", "_"))
df

,retailer,retailer_id,invoice_date,region,state,city,product,price_per_unit,units_sold,total_sales,operating_profit,operating_margin,sales_method
0,Foot Locker,1185732,2020-01-01,Northeast,New York,New York,Men's Street Footwear,50.0,1200,600000.0,300000.00,0.50,In-store
1,Foot Locker,1185732,2020-01-02,Northeast,New York,New York,Men's Athletic Footwear,50.0,1000,500000.0,150000.00,0.30,In-store
2,Foot Locker,1185732,2020-01-03,Northeast,New York,New York,Women's Street Footwear,40.0,1000,400000.0,140000.00,0.35,In-store
3,Foot Locker,1185732,2020-01-04,Northeast,New York,New York,Women's Athletic Footwear,45.0,850,382500.0,133875.00,0.35,In-store
4,Foot Locker,1185732,2020-01-05,Northeast,New York,New York,Men's Apparel,60.0,900,540000.0,162000.00,0.30,In-store
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9643,Foot Locker,1185732,2021-01-24,Northeast,New Hampshire,Manchester,Men's Apparel,50.0,64,3200.0,896.00,0.28,Outlet
9644,Foot Locker,1185732,2021-01-24,Northeast,New Hampshire,Manchester,Women's Apparel,41.0,105,4305.0,1377.60,0.32,Outlet
9645,Foot Locker,1185732,2021-02-22,Northeast,New Hampshire,Manchester,Men's Street Footwear,41.0,184,7544.0,2791.28,0.37,Outlet
9646,Foot Locker,1185732,2021-02-22,Northeast,New Hampshire,Manchester,Men's Athletic Footwear,42.0,70,2940.0,1234.80,0.42,Outlet


In [8]:
df.dtypes

retailer                    object
retailer_id                  int64
invoice_date        datetime64[ns]
region                      object
state                       object
city                        object
product                     object
price_per_unit             float64
units_sold                   int64
total_sales                float64
operating_profit           float64
operating_margin           float64
sales_method                object
dtype: object

In [9]:
df['retailer'] = df['retailer'].astype('string')

df['retailer_id'] = df['retailer_id'].astype('string')

df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors='coerce')

df[['region', 'state', 'city', 'product', 'sales_method']] = df[['region', 'state', 'city', 'product', 'sales_method']].astype('category')

float_cols = df.select_dtypes(include='float').columns
cols_not_to_round = float_cols.drop('operating_margin')

df[cols_not_to_round] = df[cols_not_to_round].round(2)

In [10]:
df.dtypes

retailer            string[python]
retailer_id         string[python]
invoice_date        datetime64[ns]
region                    category
state                     category
city                      category
product                   category
price_per_unit             float64
units_sold                   int64
total_sales                float64
operating_profit           float64
operating_margin           float64
sales_method              category
dtype: object

In [11]:
(df['units_sold'] == 0).sum()

4

In [12]:
df[df['units_sold'] == 0]

,retailer,retailer_id,invoice_date,region,state,city,product,price_per_unit,units_sold,total_sales,operating_profit,operating_margin,sales_method
1019,Foot Locker,1185732,2021-06-05,Midwest,Nebraska,Omaha,Women's Athletic Footwear,35.0,0,0.0,0.0,0.40,Outlet
1025,Foot Locker,1185732,2021-06-11,Midwest,Nebraska,Omaha,Women's Athletic Footwear,30.0,0,0.0,0.0,0.40,Outlet
4907,Foot Locker,1185732,2021-06-05,Midwest,Nebraska,Omaha,Women's Athletic Footwear,33.0,0,0.0,0.0,0.55,Online
4913,Foot Locker,1185732,2021-06-11,Midwest,Nebraska,Omaha,Women's Athletic Footwear,27.0,0,0.0,0.0,0.53,Online


In [13]:
df = df[df['units_sold'] > 0].reset_index(drop=True)

In [14]:
(df['units_sold'] == 0).sum()

0

### 4. Data Corrections

**Total Sales Anomaly:** 3,886 rows had `total_sales` values exactly 10× higher than `price_per_unit × units_sold`. These are wholesale transactions where the invoiced total did not match the listed unit price. The recalculated values were computed and retained as proof of the finding, then copied into the original `total_sales` and `operating_profit` columns before load.

**Operating Margin Conversion:** Raw decimal values (e.g. `0.30`) were converted to percentage form (e.g. `30.0`) after all profit calculations were complete, ensuring no arithmetic was affected by the scale change.

- `profit_per_unit` — operating profit divided by units sold, added to provide a per-transaction unit economics metric for analysis.

In [15]:
df.insert(10, 'recalculated_total_sales',(df['price_per_unit'] * df['units_sold']).round(2).astype('float64'))
df

,retailer,retailer_id,invoice_date,region,state,city,product,price_per_unit,units_sold,total_sales,recalculated_total_sales,operating_profit,operating_margin,sales_method
0,Foot Locker,1185732,2020-01-01,Northeast,New York,New York,Men's Street Footwear,50.0,1200,600000.0,60000.0,300000.00,0.50,In-store
1,Foot Locker,1185732,2020-01-02,Northeast,New York,New York,Men's Athletic Footwear,50.0,1000,500000.0,50000.0,150000.00,0.30,In-store
2,Foot Locker,1185732,2020-01-03,Northeast,New York,New York,Women's Street Footwear,40.0,1000,400000.0,40000.0,140000.00,0.35,In-store
3,Foot Locker,1185732,2020-01-04,Northeast,New York,New York,Women's Athletic Footwear,45.0,850,382500.0,38250.0,133875.00,0.35,In-store
4,Foot Locker,1185732,2020-01-05,Northeast,New York,New York,Men's Apparel,60.0,900,540000.0,54000.0,162000.00,0.30,In-store
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9639,Foot Locker,1185732,2021-01-24,Northeast,New Hampshire,Manchester,Men's Apparel,50.0,64,3200.0,3200.0,896.00,0.28,Outlet
9640,Foot Locker,1185732,2021-01-24,Northeast,New Hampshire,Manchester,Women's Apparel,41.0,105,4305.0,4305.0,1377.60,0.32,Outlet
9641,Foot Locker,1185732,2021-02-22,Northeast,New Hampshire,Manchester,Men's Street Footwear,41.0,184,7544.0,7544.0,2791.28,0.37,Outlet
9642,Foot Locker,1185732,2021-02-22,Northeast,New Hampshire,Manchester,Men's Athletic Footwear,42.0,70,2940.0,2940.0,1234.80,0.42,Outlet


In [16]:
(df['total_sales'] != df['recalculated_total_sales']).sum()

3886

In [17]:
df[df['total_sales'] != df['recalculated_total_sales']]

,retailer,retailer_id,invoice_date,region,state,city,product,price_per_unit,units_sold,total_sales,recalculated_total_sales,operating_profit,operating_margin,sales_method
0,Foot Locker,1185732,2020-01-01,Northeast,New York,New York,Men's Street Footwear,50.0,1200,600000.0,60000.0,300000.0,0.50,In-store
1,Foot Locker,1185732,2020-01-02,Northeast,New York,New York,Men's Athletic Footwear,50.0,1000,500000.0,50000.0,150000.0,0.30,In-store
2,Foot Locker,1185732,2020-01-03,Northeast,New York,New York,Women's Street Footwear,40.0,1000,400000.0,40000.0,140000.0,0.35,In-store
3,Foot Locker,1185732,2020-01-04,Northeast,New York,New York,Women's Athletic Footwear,45.0,850,382500.0,38250.0,133875.0,0.35,In-store
4,Foot Locker,1185732,2020-01-05,Northeast,New York,New York,Men's Apparel,60.0,900,540000.0,54000.0,162000.0,0.30,In-store
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3881,Foot Locker,1185732,2021-01-24,Northeast,New Hampshire,Manchester,Men's Apparel,65.0,200,130000.0,13000.0,32500.0,0.25,Outlet
3882,Foot Locker,1185732,2021-01-24,Northeast,New Hampshire,Manchester,Women's Apparel,55.0,300,165000.0,16500.0,49500.0,0.30,Outlet
3883,Foot Locker,1185732,2021-02-22,Northeast,New Hampshire,Manchester,Men's Street Footwear,55.0,575,316250.0,31625.0,110687.5,0.35,Outlet
3884,Foot Locker,1185732,2021-02-22,Northeast,New Hampshire,Manchester,Men's Athletic Footwear,55.0,225,123750.0,12375.0,43312.5,0.35,Outlet


In [18]:
df[df['operating_profit'].round(2) != (df['recalculated_total_sales'] * df['operating_margin']).round(2)].shape[0]

3886

In [19]:
df[df['operating_profit'].round(2) != (df['recalculated_total_sales'] * df['operating_margin']).round(2)]

,retailer,retailer_id,invoice_date,region,state,city,product,price_per_unit,units_sold,total_sales,recalculated_total_sales,operating_profit,operating_margin,sales_method
0,Foot Locker,1185732,2020-01-01,Northeast,New York,New York,Men's Street Footwear,50.0,1200,600000.0,60000.0,300000.0,0.50,In-store
1,Foot Locker,1185732,2020-01-02,Northeast,New York,New York,Men's Athletic Footwear,50.0,1000,500000.0,50000.0,150000.0,0.30,In-store
2,Foot Locker,1185732,2020-01-03,Northeast,New York,New York,Women's Street Footwear,40.0,1000,400000.0,40000.0,140000.0,0.35,In-store
3,Foot Locker,1185732,2020-01-04,Northeast,New York,New York,Women's Athletic Footwear,45.0,850,382500.0,38250.0,133875.0,0.35,In-store
4,Foot Locker,1185732,2020-01-05,Northeast,New York,New York,Men's Apparel,60.0,900,540000.0,54000.0,162000.0,0.30,In-store
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3881,Foot Locker,1185732,2021-01-24,Northeast,New Hampshire,Manchester,Men's Apparel,65.0,200,130000.0,13000.0,32500.0,0.25,Outlet
3882,Foot Locker,1185732,2021-01-24,Northeast,New Hampshire,Manchester,Women's Apparel,55.0,300,165000.0,16500.0,49500.0,0.30,Outlet
3883,Foot Locker,1185732,2021-02-22,Northeast,New Hampshire,Manchester,Men's Street Footwear,55.0,575,316250.0,31625.0,110687.5,0.35,Outlet
3884,Foot Locker,1185732,2021-02-22,Northeast,New Hampshire,Manchester,Men's Athletic Footwear,55.0,225,123750.0,12375.0,43312.5,0.35,Outlet


In [20]:
df.insert(12, 'recalculated_operating_profit',(df['recalculated_total_sales'] * df['operating_margin']).round(2))
df

,retailer,retailer_id,invoice_date,region,state,city,product,price_per_unit,units_sold,total_sales,recalculated_total_sales,operating_profit,recalculated_operating_profit,operating_margin,sales_method
0,Foot Locker,1185732,2020-01-01,Northeast,New York,New York,Men's Street Footwear,50.0,1200,600000.0,60000.0,300000.00,30000.00,0.50,In-store
1,Foot Locker,1185732,2020-01-02,Northeast,New York,New York,Men's Athletic Footwear,50.0,1000,500000.0,50000.0,150000.00,15000.00,0.30,In-store
2,Foot Locker,1185732,2020-01-03,Northeast,New York,New York,Women's Street Footwear,40.0,1000,400000.0,40000.0,140000.00,14000.00,0.35,In-store
3,Foot Locker,1185732,2020-01-04,Northeast,New York,New York,Women's Athletic Footwear,45.0,850,382500.0,38250.0,133875.00,13387.50,0.35,In-store
4,Foot Locker,1185732,2020-01-05,Northeast,New York,New York,Men's Apparel,60.0,900,540000.0,54000.0,162000.00,16200.00,0.30,In-store
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9639,Foot Locker,1185732,2021-01-24,Northeast,New Hampshire,Manchester,Men's Apparel,50.0,64,3200.0,3200.0,896.00,896.00,0.28,Outlet
9640,Foot Locker,1185732,2021-01-24,Northeast,New Hampshire,Manchester,Women's Apparel,41.0,105,4305.0,4305.0,1377.60,1377.60,0.32,Outlet
9641,Foot Locker,1185732,2021-02-22,Northeast,New Hampshire,Manchester,Men's Street Footwear,41.0,184,7544.0,7544.0,2791.28,2791.28,0.37,Outlet
9642,Foot Locker,1185732,2021-02-22,Northeast,New Hampshire,Manchester,Men's Athletic Footwear,42.0,70,2940.0,2940.0,1234.80,1234.80,0.42,Outlet


In [21]:
df.insert(9, 'profit_per_unit',(df['recalculated_operating_profit'] / df['units_sold']).round(2).astype('float64'))
df

,retailer,retailer_id,invoice_date,region,state,city,product,price_per_unit,units_sold,profit_per_unit,total_sales,recalculated_total_sales,operating_profit,recalculated_operating_profit,operating_margin,sales_method
0,Foot Locker,1185732,2020-01-01,Northeast,New York,New York,Men's Street Footwear,50.0,1200,25.00,600000.0,60000.0,300000.00,30000.00,0.50,In-store
1,Foot Locker,1185732,2020-01-02,Northeast,New York,New York,Men's Athletic Footwear,50.0,1000,15.00,500000.0,50000.0,150000.00,15000.00,0.30,In-store
2,Foot Locker,1185732,2020-01-03,Northeast,New York,New York,Women's Street Footwear,40.0,1000,14.00,400000.0,40000.0,140000.00,14000.00,0.35,In-store
3,Foot Locker,1185732,2020-01-04,Northeast,New York,New York,Women's Athletic Footwear,45.0,850,15.75,382500.0,38250.0,133875.00,13387.50,0.35,In-store
4,Foot Locker,1185732,2020-01-05,Northeast,New York,New York,Men's Apparel,60.0,900,18.00,540000.0,54000.0,162000.00,16200.00,0.30,In-store
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9639,Foot Locker,1185732,2021-01-24,Northeast,New Hampshire,Manchester,Men's Apparel,50.0,64,14.00,3200.0,3200.0,896.00,896.00,0.28,Outlet
9640,Foot Locker,1185732,2021-01-24,Northeast,New Hampshire,Manchester,Women's Apparel,41.0,105,13.12,4305.0,4305.0,1377.60,1377.60,0.32,Outlet
9641,Foot Locker,1185732,2021-02-22,Northeast,New Hampshire,Manchester,Men's Street Footwear,41.0,184,15.17,7544.0,7544.0,2791.28,2791.28,0.37,Outlet
9642,Foot Locker,1185732,2021-02-22,Northeast,New Hampshire,Manchester,Men's Athletic Footwear,42.0,70,17.64,2940.0,2940.0,1234.80,1234.80,0.42,Outlet


In [22]:
df['total_sales'] = df['recalculated_total_sales']
df['operating_profit'] = df['recalculated_operating_profit']
df.drop(columns=['recalculated_total_sales', 'recalculated_operating_profit'], inplace=True)
df['operating_margin'] = (df['operating_margin'] * 100).round(2)
df

,retailer,retailer_id,invoice_date,region,state,city,product,price_per_unit,units_sold,profit_per_unit,total_sales,operating_profit,operating_margin,sales_method
0,Foot Locker,1185732,2020-01-01,Northeast,New York,New York,Men's Street Footwear,50.0,1200,25.00,60000.0,30000.00,50.0,In-store
1,Foot Locker,1185732,2020-01-02,Northeast,New York,New York,Men's Athletic Footwear,50.0,1000,15.00,50000.0,15000.00,30.0,In-store
2,Foot Locker,1185732,2020-01-03,Northeast,New York,New York,Women's Street Footwear,40.0,1000,14.00,40000.0,14000.00,35.0,In-store
3,Foot Locker,1185732,2020-01-04,Northeast,New York,New York,Women's Athletic Footwear,45.0,850,15.75,38250.0,13387.50,35.0,In-store
4,Foot Locker,1185732,2020-01-05,Northeast,New York,New York,Men's Apparel,60.0,900,18.00,54000.0,16200.00,30.0,In-store
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9639,Foot Locker,1185732,2021-01-24,Northeast,New Hampshire,Manchester,Men's Apparel,50.0,64,14.00,3200.0,896.00,28.0,Outlet
9640,Foot Locker,1185732,2021-01-24,Northeast,New Hampshire,Manchester,Women's Apparel,41.0,105,13.12,4305.0,1377.60,32.0,Outlet
9641,Foot Locker,1185732,2021-02-22,Northeast,New Hampshire,Manchester,Men's Street Footwear,41.0,184,15.17,7544.0,2791.28,37.0,Outlet
9642,Foot Locker,1185732,2021-02-22,Northeast,New Hampshire,Manchester,Men's Athletic Footwear,42.0,70,17.64,2940.0,1234.80,42.0,Outlet


In [23]:
df.dtypes

retailer            string[python]
retailer_id         string[python]
invoice_date        datetime64[ns]
region                    category
state                     category
city                      category
product                   category
price_per_unit             float64
units_sold                   int64
profit_per_unit            float64
total_sales                float64
operating_profit           float64
operating_margin           float64
sales_method              category
dtype: object

In [24]:
df.describe()

,invoice_date,price_per_unit,units_sold,profit_per_unit,total_sales,operating_profit,operating_margin
count,9644,9644.000000,9644.000000,9644.000000,9644.000000,9644.000000,9644.00000
mean,2021-05-10 15:03:48.452923904,45.222418,257.036603,18.931536,12460.249896,4896.823693,42.29718
min,2020-01-01 00:00:00,7.000000,6.000000,2.940000,160.000000,75.200000,10.00000
25%,2021-02-17 00:00:00,35.000000,106.000000,13.920000,4068.000000,1753.965000,35.00000
50%,2021-06-04 00:00:00,45.000000,176.000000,18.000000,7812.000000,3264.000000,41.00000
75%,2021-09-16 00:00:00,55.000000,350.000000,22.960000,15872.000000,6193.830000,49.00000
max,2021-12-31 00:00:00,110.000000,1275.000000,57.680000,82500.000000,39000.000000,80.00000
std,NaN,14.705565,214.232536,7.328289,12716.498341,4866.452163,9.72023


In [25]:
df.isnull().sum()

retailer            0
retailer_id         0
invoice_date        0
region              0
state               0
city                0
product             0
price_per_unit      0
units_sold          0
profit_per_unit     0
total_sales         0
operating_profit    0
operating_margin    0
sales_method        0
dtype: int64

In [26]:
df.isin(['UNKNOWN', 'ERROR', '']).sum()

retailer            0
retailer_id         0
invoice_date        0
region              0
state               0
city                0
product             0
price_per_unit      0
units_sold          0
profit_per_unit     0
total_sales         0
operating_profit    0
operating_margin    0
sales_method        0
dtype: int64

In [27]:
unique_values = ['total_sales', 'operating_profit', 'price_per_unit']

for x in unique_values:
    print(f"\nColumn_name: {x}")
    print(df[x].unique())


Column_name: total_sales
[60000. 50000. 40000. ...  4902.  5664.  8784.]

Column_name: operating_profit
[30000.   15000.   14000.   ...   896.    2791.28   649.89]

Column_name: price_per_unit
[ 50.  40.  45.  60.  55.  65.  70.  25.  35.  30.  80.  75.  20.  85.
 100.  90.  95.  15.  10. 110. 105.  47.  36.  41.  46.  44.  58.  48.
  39.  43.  59.  56.  49.  54.  64.  53.  61.  62.  68.  52.  66.  51.
  24.  34.  33.  23.  32.  38.  29.  37.  27.  42.  72.  63.  69.  76.
  67.  57.  28.  18.  19.  71.  78.  73.  74.  83.  82.  98.  77.  88.
  86.  14.   9.  97.  81.  79.  96.  84.  89. 103. 101.  87.  92.  31.
  26.  21.  13.  22.  17.  12.  16.  11.   7.   8.]


In [28]:
unique_values = ['operating_margin']

for x in unique_values:
    print(f"\nColumn_name: {x}")
    print(df[x].unique())


Column_name: operating_margin
[50. 30. 35. 25. 45. 20. 15. 40. 55. 10. 60. 65. 61. 42. 46. 37. 62. 44.
 47. 49. 43. 38. 63. 41. 64. 48. 36. 39. 58. 59. 57. 31. 28. 27. 51. 66.
 33. 34. 29. 24. 23. 21. 52. 53. 68. 69. 67. 54. 56. 70. 74. 26. 76. 77.
 73. 32. 75. 72. 71. 80. 79. 22. 19. 17. 18. 12.]


In [29]:
unique_values = ['retailer', 'retailer_id', 'region', 'state', 'city', 'product', 'price_per_unit', 'sales_method']

for x in unique_values:
    print(f"\nColumn_name: {x}")
    print(df[x].unique())


Column_name: retailer
<StringArray>
['Foot Locker', 'Walmart', 'Sports Direct', 'West Gear', "Kohl's", 'Amazon']
Length: 6, dtype: string

Column_name: retailer_id
<StringArray>
['1185732', '1197831', '1128299', '1189833']
Length: 4, dtype: string

Column_name: region
['Northeast', 'South', 'West', 'Midwest', 'Southeast']
Categories (5, object): ['Midwest', 'Northeast', 'South', 'Southeast', 'West']

Column_name: state
['New York', 'Texas', 'California', 'Illinois', 'Pennsylvania', ..., 'Connecticut', 'Rhode Island', 'Massachusetts', 'Vermont', 'New Hampshire']
Length: 50
Categories (50, object): ['Alabama', 'Alaska', 'Arizona', 'Arkansas', ..., 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming']

Column_name: city
['New York', 'Houston', 'San Francisco', 'Los Angeles', 'Chicago', ..., 'Hartford', 'Providence', 'Boston', 'Burlington', 'Manchester']
Length: 52
Categories (52, object): ['Albany', 'Albuquerque', 'Anchorage', 'Atlanta', ..., 'Sioux Falls', 'St. Louis', 'Wichita', 'Wilm

### 5. Validation

A `validate()` function asserts four invariants before any data is exported:
- `total_sales ≥ 0` for all rows
- `units_sold > 0` for all rows
- `operating_margin` between 0 and 100
- No null values in `invoice_date`

In [30]:
def validate(df):
    assert df['total_sales'].ge(0).all(),                "Negative total_sales found"
    assert df['units_sold'].gt(0).all(),                 "Zero or negative units_sold found"
    assert df['operating_margin'].between(0, 100).all(), "Margin outside expected 0–100% range"
    assert df['invoice_date'].notna().all(),             "Null values found in invoice_date"
    print("All validation checks passed.")

validate(df)

All validation checks passed.


In [31]:
df.to_excel("cleaned_adidas_US_sales_dataset.xlsx", index=False)

### 6. Loading to MySQL
- Cleaned DataFrame loaded to MySQL via SQLAlchemy (`mysql+pymysql`)
- Database: `adidas_data`; Table: adidas
- The table is replaced on each run to ensure fresh data.
- Load verified with a `SELECT *` query confirming row count and schema

In [32]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

username = 'YOUR_USERNAME'
password = 'YOUR_USERNAME' 
host = 'localhost'
port = 3307
database = 'adidas_data'

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")
try:
    with engine.connect() as conn:
        print("Connection successful!")
except Exception as e:
    print(f"Connection failed: {e}")

df.to_sql(
    name='adidas',
    con=engine,
    if_exists='replace',
    index=False
)

print("Data successfully loaded!")

Connection successful!
Data successfully loaded!


In [33]:
query = "SELECT COUNT(*) AS total_rows FROM adidas"
result = pd.read_sql(query, engine)
print(f"Rows loaded to MySQL: {result['total_rows'][0]:,}")

Rows loaded to MySQL: 9,644


### 7. Verification
Reading the data back from MySQL to confirm the pipeline loaded successfully and all 14 columns are intact.

Final cleaned dataset:9644 rows × 14 columns

Records dropped during cleaning: 4 rows

New column added: profit_per_unit